# LogStruct: Cell Type Classification with Network Priors

This notebook demonstrates using LogStruct for cell type classification with a gene network prior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

from logstruct import LogStructClassifier
from logstruct import priors

## Generate Synthetic Data

We'll create data where features are organized into correlated groups (mimicking pathways).

In [ ]:
np.random.seed(42)

n_samples = 500
n_features = 100
n_groups = 10  # "pathways"
features_per_group = n_features // n_groups

# Generate correlated features within groups
X = np.zeros((n_samples, n_features))
for g in range(n_groups):
    start = g * features_per_group
    end = start + features_per_group
    
    # Shared latent factor + noise
    latent = np.random.randn(n_samples, 1)
    noise = 0.5 * np.random.randn(n_samples, features_per_group)
    X[:, start:end] = latent + noise

# Labels based on first two groups
y = ((X[:, 0] + X[:, 10]) > 0).astype(int)

print(f"Data shape: {X.shape}")
print(f"Class balance: {np.mean(y):.2f}")

## Build Prior Adjacency

We'll create a prior that reflects the true group structure.

In [ ]:
# Build edges: features in same group are connected
edges = []
for g in range(n_groups):
    start = g * features_per_group
    end = start + features_per_group
    for i in range(start, end):
        for j in range(i + 1, end):
            edges.append((i, j, 0.7))  # within-group weight

prior = priors.from_edge_list(n_features, edges, baseline=0.05)

stats = priors.summarize(prior)
print(f"Prior stats: {stats}")

In [ ]:
# Visualize prior
fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(prior, cmap='Blues', ax=ax, cbar_kws={'label': 'Edge weight'})
ax.set_title('Prior Adjacency Matrix')
ax.set_xlabel('Feature')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

## Train LogStruct Classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

clf = LogStructClassifier(
    prior_adjacency=prior,
    lambda_en=1.0,
    lambda_smooth=2.0,   # encourage similar coefficients for connected features
    lambda_kl=1.0,       # trust the prior moderately
    alpha=0.5,           # elastic net mix
    max_iter=100,
    verbose=True,
    random_state=42,
)

clf.fit(X_train, y_train)

## Evaluate

In [ ]:
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print(f"Test accuracy: {clf.score(X_test, y_test):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(clf.history_['train_loss'], label='Train')
axes[0].plot(clf.history_['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_title('Loss Curves')

axes[1].plot(clf.history_['train_acc'], label='Train')
axes[1].plot(clf.history_['val_acc'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_title('Accuracy Curves')

plt.tight_layout()
plt.show()

## Examine Learned Structure

In [ ]:
# Coefficients
coef = clf.coef_

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(coef)), coef)
ax.set_xlabel('Feature')
ax.set_ylabel('Coefficient')
ax.set_title('Learned Coefficients')

# Highlight feature groups
for g in range(n_groups):
    start = g * features_per_group
    ax.axvline(start, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Learned adjacency
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(prior, cmap='Blues', ax=axes[0], cbar_kws={'label': 'Weight'})
axes[0].set_title('Prior Adjacency')

sns.heatmap(clf.adjacency_, cmap='Blues', ax=axes[1], cbar_kws={'label': 'Weight'})
axes[1].set_title('Learned Adjacency')

plt.tight_layout()
plt.show()

In [ ]:
# Top learned edges
print("Top 15 learned edges:")
print("-" * 40)
for i, j, w in clf.get_top_edges(15):
    group_i = i // features_per_group
    group_j = j // features_per_group
    same_group = "same" if group_i == group_j else "diff"
    print(f"Feature {i:3d} -- Feature {j:3d}: {w:.3f}  (groups: {group_i}, {group_j} = {same_group})")

## Compare: With vs Without Network Prior

In [ ]:
from sklearn.linear_model import LogisticRegression

# Standard logistic regression
lr = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, max_iter=1000)
lr.fit(X_train, y_train)

# LogStruct with uninformative prior
uninformative_prior = priors.from_identity(n_features, baseline=0.1)
clf_no_prior = LogStructClassifier(
    prior_adjacency=uninformative_prior,
    lambda_smooth=0.0,  # no smoothing
    lambda_kl=0.0,      # no KL
    max_iter=100,
    random_state=42,
)
clf_no_prior.fit(X_train, y_train)

print("Comparison:")
print(f"  sklearn LogisticRegression: {lr.score(X_test, y_test):.3f}")
print(f"  LogStruct (no prior):       {clf_no_prior.score(X_test, y_test):.3f}")
print(f"  LogStruct (with prior):     {clf.score(X_test, y_test):.3f}")

## Summary

LogStruct provides:
1. **Coefficients** that respect network structure (connected features have similar weights)
2. **Learned adjacency** that refines the prior based on predictive signal
3. **Interpretable edges** showing which feature relationships matter for prediction

This is most valuable when:
- You have reliable prior knowledge (pathways, PPI)
- Sample size is limited
- Interpretability matters more than raw accuracy